<a href="https://colab.research.google.com/github/HimanshuJha0005/Machine-Learning-Internship-Projects/blob/main/Movie_Genre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# 1. Load the dataset (with exact path)
import pandas as pd
train_data = pd.read_csv('/content/train_data.txt', sep=':::', engine='python', names=['ID', 'TITLE', 'GENRE', 'DESCRIPTION'])

In [3]:
import os
import re
import nltk
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

# Download NLTK stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
# 1. Load the training dataset
try:
    train_data = pd.read_csv('train_data.txt', sep=':::', engine='python', names=['ID', 'TITLE', 'GENRE', 'DESCRIPTION'])
    print("Training data loaded successfully!")
except FileNotFoundError:
    print("Error: 'train_data.txt' not found. Please upload the file to your directory.")

# 2. Text Cleaning Function
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    words = text.split()
    cleaned_words = [word for word in words if word not in stop_words]  # Remove stopwords
    return " ".join(cleaned_words)

# Apply cleaning to the description column
print("Cleaning movie descriptions in training data... (This may take a moment)")
train_data['CLEAN_DESCRIPTION'] = train_data['DESCRIPTION'].apply(clean_text)
print("Cleaning complete!")

Training data loaded successfully!
Cleaning movie descriptions in training data... (This may take a moment)
Cleaning complete!


In [ ]:
# 1. Initialize TF-IDF Vectorizer[cite: 2]
tfidf = TfidfVectorizer(max_features=10000)
X = tfidf.fit_transform(train_data['CLEAN_DESCRIPTION'])
y = train_data['GENRE']

# 2. Split dataset into Training and Validation sets (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the Naive Bayes Model[cite: 2]
print("Training the Multinomial Naive Bayes model...")
model = MultinomialNB()
model.fit(X_train, y_train)
print("Model training complete!")

# 4. Check validation accuracy
y_pred = model.predict(X_val)
print("Validation Accuracy Score:", accuracy_score(y_val, y_pred))

Training the Multinomial Naive Bayes model...
Model training complete!
Validation Accuracy Score: 0.513879922530665


In [ ]:
def predict_genre(plot_summary):
    cleaned_plot = clean_text(plot_summary)
    vectorized_plot = tfidf.transform([cleaned_plot])
    prediction = model.predict(vectorized_plot)
    return prediction[0]

# Quick manual tests
print("\n--- Testing Custom Stories ---")
story_1 = "A spacecraft travels through a wormhole in search of a new home for humanity."
print(f"Story: {story_1}")
print(f"Predicted Genre: {predict_genre(story_1).upper()}")

story_2 = "A family moves into a haunted house where dark spirits hunt them at night."
print(f"Story: {story_2}")
print(f"Predicted Genre: {predict_genre(story_2).upper()}")


--- Testing Custom Stories ---
Story: A spacecraft travels through a wormhole in search of a new home for humanity.
Predicted Genre:  DOCUMENTARY 
Story: A family moves into a haunted house where dark spirits hunt them at night.
Predicted Genre:  HORROR 


In [ ]:
# Predict on the official test data file
if 'test_data.txt' in os.listdir('.'):
    # 1. Load test dataset
    test_data = pd.read_csv('test_data.txt', sep=':::', engine='python', names=['ID', 'TITLE', 'DESCRIPTION'])
    print("Test data loaded successfully!")

    # 2. Clean test dataset (Using a subset of 1000 rows for quick execution)
    print("Cleaning test descriptions (Subset of 1000 rows for quick test)...")
    test_subset = test_data.head(1000).copy()
    test_subset['CLEAN_DESCRIPTION'] = test_subset['DESCRIPTION'].apply(clean_text)

    # 3. Vectorize and Predict
    X_test = tfidf.transform(test_subset['CLEAN_DESCRIPTION'])
    test_subset['PREDICTED_GENRE'] = model.predict(X_test)

    # 4. Display Results
    print("\n--- Predictions on Test Data ---")
    print(test_subset[['TITLE', 'PREDICTED_GENRE']].head(10))
else:
    print("Error: 'test_data.txt' not found. Please upload the test file to run batch predictions.")

Test data loaded successfully!
Cleaning test descriptions (Subset of 1000 rows for quick test)...

--- Predictions on Test Data ---
                                          TITLE PREDICTED_GENRE
0                         Edgar's Lunch (1998)           drama 
1                     La guerra de papá (1977)           drama 
2                  Off the Beaten Track (2010)     documentary 
3                       Meu Amigo Hindu (2015)           drama 
4                            Er nu zhai (1955)           drama 
5                           Riddle Room (2016)           drama 
6                               L'amica (1969)           drama 
7                         Ina Mina Dika (1989)           drama 
8   Equinox Special: Britain's Tornados (2005)     documentary 
9                                 Press (2011)           drama 


In [ ]:
# 1. Load the actual test solutions
if 'test_data_solution.txt' in os.listdir('.'):
    test_solutions = pd.read_csv('test_data_solution.txt', sep=':::', engine='python', names=['ID', 'TITLE', 'GENRE', 'DESCRIPTION'])
    print("Test solutions loaded successfully!")

    # 2. Extract actual and predicted genres for comparison (matching the 1000 rows subset)
    actual_genres = test_solutions['GENRE'].head(1000).str.strip().tolist()
    predicted_genres = test_subset['PREDICTED_GENRE'].str.strip().tolist()

    # 3. Calculate and display final evaluation metrics
    test_accuracy = accuracy_score(actual_genres, predicted_genres)

    print("\n--- Final Test Evaluation ---")
    print(f"Accuracy on Test Data: {test_accuracy * 100:.2f}%")
    print("\nDetailed Classification Report:")
    print(classification_report(actual_genres, predicted_genres, zero_division=0))
else:
    print("Error: 'test_data_solution.txt' is not in the directory. Please upload it to run the evaluation.")

Test solutions loaded successfully!

--- Final Test Evaluation ---
Accuracy on Test Data: 52.30%

Detailed Classification Report:
              precision    recall  f1-score   support

      action       0.00      0.00      0.00        21
       adult       0.00      0.00      0.00         6
   adventure       0.00      0.00      0.00         8
   animation       0.00      0.00      0.00        10
   biography       0.00      0.00      0.00         8
      comedy       0.48      0.33      0.39       143
       crime       0.00      0.00      0.00         7
 documentary       0.60      0.87      0.71       236
       drama       0.46      0.89      0.61       271
      family       0.00      0.00      0.00        13
     fantasy       0.00      0.00      0.00         6
   game-show       0.00      0.00      0.00         5
     history       0.00      0.00      0.00         6
      horror       0.85      0.29      0.43        38
       music       0.00      0.00      0.00        17
     